## Working with synthetic banking data

In [1]:
import pyspark
from pyspark.sql import SparkSession


In [2]:
spark = SparkSession.builder \
    .appName("Synthetic_data") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()
spark

In [3]:
import time 

### Reading the datasets

In [4]:
### Customer data
start_time = time.time()
customer_df = spark.read.csv(r'../datasets/data/customers.csv', header = True, inferSchema=True)
end_time = time.time()
print("Time taken:", end_time - start_time, "seconds")

Time taken: 5.284194469451904 seconds


In [5]:
customer_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_type: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- customer_category: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- date_of_birth: date (nullable = true)
 |-- marital_status: string (nullable = true)
 |-- nationality: string (nullable = true)
 |-- country_of_birth: string (nullable = true)
 |-- residential_country: string (nullable = true)
 |-- residence_state: string (nullable = true)
 |-- tax_residence_country: string (nullable = true)
 |-- is_non_resident: boolean (nullable = true)
 |-- id_document_type: string (nullable = true)
 |-- id_issuing_country: string (nullable = true)
 |-- occupation: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- employment_status: string (nullable = true)
 |-- education: string (nullable = true)
 |-- declared_annual_income: double (nul

In [6]:
print(customer_df.count(),",", len(customer_df.columns))

102000 , 33


In [7]:
### account data
start_time = time.time()
account_df = spark.read.csv(r'../datasets/data/accounts.csv', header = True, inferSchema=True)
end_time = time.time()
print("Time taken:", end_time - start_time, "seconds")

Time taken: 1.7210392951965332 seconds


In [8]:
account_df.printSchema()

root
 |-- account_id: string (nullable = true)
 |-- linked_customer_id: string (nullable = true)
 |-- account_type: string (nullable = true)
 |-- product_subtype: string (nullable = true)
 |-- product_code: string (nullable = true)
 |-- currency_code: string (nullable = true)
 |-- account_opening_date: date (nullable = true)
 |-- account_status: string (nullable = true)
 |-- account_ownership_type: string (nullable = true)
 |-- account_purpose: string (nullable = true)
 |-- account_tenure_months: integer (nullable = true)
 |-- credit_limit: double (nullable = true)
 |-- current_balance: double (nullable = true)
 |-- account_opening_channel: string (nullable = true)
 |-- branch_code: string (nullable = true)
 |-- branch_location: string (nullable = true)
 |-- interest_bearing: string (nullable = true)



In [9]:
print(account_df.count(),",", len(account_df.columns))

205616 , 17


In [10]:
### Transaction data
start_time = time.time()
transaction_df = spark.read.csv(r'../datasets/data/transactions.csv', header = True, inferSchema=True)
end_time = time.time()
print("Time taken:", end_time - start_time, "seconds")

Time taken: 29.82299280166626 seconds


In [11]:
print(transaction_df.count(),",", len(transaction_df.columns))

8247654 , 66


### Merging the 3 datasets

In [12]:
from pyspark.sql import functions as F
from pyspark.storagelevel import StorageLevel
import time
import gc

In [13]:

## renaming the column before merging
account_df = account_df.withColumnRenamed(
    "linked_customer_id",
    "customer_id"
)


In [14]:
## Merging customer and account dataset
start_time = time.time()

customers_accounts_df = account_df.join(
    F.broadcast(customer_df),
    on="customer_id",
    how="left"
)

# Persist because this dataframe will be reused
customers_accounts_df.persist(StorageLevel.MEMORY_AND_DISK)

# Triggering execution
customers_accounts_df.count()

end_time = time.time()

print(f"Customer-Account merge completed in {end_time - start_time:.2f} seconds")


Customer-Account merge completed in 7.06 seconds


In [15]:
## Removing unused dfs to free memory
del customer_df
del account_df
gc.collect()


157

In [16]:
## Merging transaction with customer account data

start_time = time.time()
transaction_df = transaction_df.drop("customer_id")
tms_df = transaction_df.join(
    customers_accounts_df,
    on="account_id",
    how="left"
)

# Persist final dataframe 
tms_df.persist(StorageLevel.MEMORY_AND_DISK)

# Trigger execution
tms_df.show(5)

end_time = time.time()

print(f"Final merge completed in {end_time - start_time:.2f} seconds")


+-------------+--------------+--------------------+-------------------+-------------------+------------------+----------------------+----------------+--------------------+--------------------+-------------------+----------------------+-------------+-------------+----------------+----------------+---------------------+----------+-------------+------------+-------------------+--------------------+-----------------------+---------------------+------------------------+----------------------+-----------------+---------------+--------------+-----------------+--------------------+----------------------+--------------------+------------+-------------+-------------+--------+--------------------+-----------------+--------------------+---------------------+----------------+-------------------+----------------------+------------------+------------------+------------------------+-----------------------+---------------------------+------------------------+-----------------------+--------------------

In [17]:
## removing unused dataframes
del transaction_df
del customers_accounts_df
gc.collect()

tms_df.printSchema()


root
 |-- account_id: string (nullable = true)
 |-- transaction_id: string (nullable = true)
 |-- transaction_datetime: timestamp (nullable = true)
 |-- posting_date: timestamp (nullable = true)
 |-- value_date: timestamp (nullable = true)
 |-- is_training_window: boolean (nullable = true)
 |-- debit_credit_indicator: string (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- transaction_sub_type: string (nullable = true)
 |-- transaction_category: string (nullable = true)
 |-- transaction_channel: string (nullable = true)
 |-- transaction_amount_usd: double (nullable = true)
 |-- orig_currency: string (nullable = true)
 |-- bene_currency: string (nullable = true)
 |-- orig_curr_amount: double (nullable = true)
 |-- bene_curr_amount: double (nullable = true)
 |-- exchange_rate_applied: double (nullable = true)
 |-- fee_amount: double (nullable = true)
 |-- orig_account: string (nullable = true)
 |-- bene_account: string (nullable = true)
 |-- originator_bank_bic: str

In [18]:
print(f"Final dataset shape: {tms_df.count()},{len(tms_df.columns)}")

Final dataset shape: 8248270,113


In [19]:
## Saving the final merged df

start_time = time.time()

tms_df.write \
    .mode("overwrite") \
    .parquet("../datasets/data/final_merged_data_parquet")

end_time = time.time()

print(f"Parquet save completed in {end_time - start_time:.2f} seconds")

Py4JJavaError: An error occurred while calling o79.parquet.
: java.lang.RuntimeException: java.io.FileNotFoundException: java.io.FileNotFoundException: Hadoop home directory C:\hadoop\bin does not exist -see https://cwiki.apache.org/confluence/display/HADOOP2/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:789)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:298)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:314)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:1179)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:861)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:901)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:873)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:900)
	at org.apache.hadoop.fs.RawLocalFileSyste

In [20]:
## Error coming since hadoop is not configured